# CS156: Pipeline - First Draft

**Do I sound like AI, or have I always been a bad writer?**  

With the amount of AI we use on a day to day, I am fairly sure that a lot of the content I consume, whether it be the news, social media, or even the PCW that I grade, might be AI-assisted, if not completely AI-generated. This makes me wonder: as a symptom of consuming so much 'slop,' could my own writing also be showing traits of AI-generated text? 

This report answers this question by first training a model to classify assignments I wrote pre-Fall 2024, when AI tools have shown enough progress to make using it as an aid in coursework a viable strategy, from later submissions. Then, I use a (INSERT WHAT I ACTUALLY DO BASED ON PRELIMINARY MODEL RESULTS). 


## The Data

To create the dataset to be used in this pipeline, I retrieved past written assignments stored in my Minerva email's Google Drive using the notebook here ran in Colab ([GitHub](https://colab.research.google.com/drive/1f_YmZ3cR82UPcLH7uNJ9DUu-x7Gllc7c#scrollTo=e160UdxwwBmt)). Prior to the retreival, I manually labelled the documents that I wish to include. 

Importantly, since the focus of the models trained here is stylistic drift in *my* writing, I excluded the following assignments: 
- **Skill builders from CS111 and CS113:** The content is mostly mathematical equations and instructions, and they do not contain much original writing. In addition, they are hand-written documents, which adds more complexity to the ingestion process. 
- **Group assignments:** These contain writing by other students, thus were excluded to limit the training data to my own writing. 
- **Heavily templated assignments:** All assignments formatted as workbooks (both Forum codebooks and worksheets with instructions included) were excluded to prevent 'dilution' of my writing due to the instruction text included in the document. This resulted in the exclusion of several FA50 and FA51 (previously CS50 and CS51) assignments, Deep Dives from CS111 and CS113, problems sets from CS114, and some SS110 and SS152 assginments that are formatted as cover sheets. 

For the same reasons, I included any non-essay documents that reflect my own prose, such as pre-written scripts for any videos I submitted as assignments, or reflection documents that supplement technical interviews. 

Based on these exclusion criteria, 55 assignment documents qualified for inclusion in the dataset. However, 4 of these assignments were created outside of Google Drive (e.g., PDFs exported from OverLeaf), and I excluded them from the final dataset. Extracting text from PDFs would require different data processing compared to Google Drive documents, and given the small number of such files, I chose to maintain consistency by excluding them entirely.

## Loading the Data

Using (HOW DO I SCRAPE GDRIVE LOOK THIS UP), I loaded the text data from my past assignments in to a pandas dataframe and tokenized (UPDATE AS I ADD CODE). 

In [1]:
# Install necessary packages
!pip install pandas numpy scikit-learn matplotlib seaborn scipy textstat nltk


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import re

# Sklearn imports
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.decomposition import PCA

# Stylometric analysis
import textstat
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from collections import Counter

In [3]:
# Loading dataset

df = pd.read_csv("gdrive_scraped_data.csv", encoding='utf-8') 


## Pre-processing and EDA

While I am most interested in stylistic drift, I first train a baseline classifier (WHICH) to see whether there is any meaningful difference between my pre-AI (pre-Fall 2024) and post-AI (Fall 2024 and later) writing.

This section describes the exact cleaning steps used in the code and the assumptions behind them.

### Data Cleaning to Prevent Leakage

To ensure the classifier learns stylistic patterns rather than metadata, I wanted to train it on cleaned data where elements or sections that may cause leakage are removed. In order to compare model performance across data with different levels of cleaning to verify the model is truly learning from the main text, I created three text columns by applying a fixed cleaning pipeline: `remove_cover_sheet` -> `remove_toc` -> `extract_body_only` -> `remove_hc_tags`.

- `text_raw`: Original text with all content. This is pulled directly from the source csv file. 
- `no_cover`: Cover sheet (or equivalent) removed. This strips the course code, instructor name, and deadline, which can cause leakage since all correlate with course timing and therefore pre/post AI labels. As an artefact of this processing the institution name (Minerva University) was also removed. I also chose to remove table of contents when present in order to maintain internal consistency, as these do not carry any stylistic signal.  
- `text_clean`: Fully cleaned. In addition to the cover sheet, I also removed References, Appendices, and HC/LO tags, which can cause leakage from course/topic/time. I also removed AI Statements because the wording, tool names, and format may shift over time or by course. This is justified as the goal is to capture main-body prose, not these sections, that are less stylistic and more structural. 

In addition to cleaning, I added the following metadata columns: 
- `course_code`: Extracted from the cover sheet using a 2-letter + 2-3 digit regex. 
- `label`: The label for model training. Pre-AI work is represented by 0 and post-AI work is represented by 1. 

Importantly, each cleaning step makes some key assumptions about my data, which I verified as I implemented the code: 

- **Cover sheets exist at the start of the document.** The cover sheet is assumed to appear in the first 500-3000 characters and to contain either a Minerva header, a horizontal rule, a large page break, or a course code. This lets the heuristic remove only the top portion.
- **TOCs are near the top and look like lists with page numbers.** The TOC removal checks the first ~20 lines for multiple lines ending in digits or an explicit "Contents" header.
- **End-matter starts at explicit markers.** The body is truncated at the earliest of: References/Bibliography, Appendix/Appendices, Word Count, or AI Statement/AI Use headings, but only if they appear after 100+ characters.
- **HC/LO tags are structured as #tag: blocks.** The cleaning assumes these are metadata blocks that can be removed without losing core prose.

This setup lets me compare model performance on raw text vs. main-body prose to check whether the model is learning writing style rather than metadata.


In [ ]:
# Preprocessing functions

VALID_COURSES = ['CP192', 'CS146', 'GL96', 'CS156', 'SS111',
                 'CP191', 'CS166', 'GL95', 'SS152', 'SS156', 
                 'CS113', 'CS114', 'GL94', 'SS110', 
                 'CS110', 'CS111', 'GL93', 'SS112', 
                 'CX51', 'EA51', 'FA51', 'GL92', 'MC51', 
                 'CX50', 'EA50', 'FA50', 'GL91', 'MC50', 'IL199']

def extract_course_code(text):
    """
    Extract course code from cover sheet (e.g., CS110, GL96).
    
    Searches first 500 chars for 2-letter prefix + 2-3 digit number.
    Returns: course_code or None if not found
    """
    pattern = r'\b([A-Z]{2})[\s-]?(\d{2,3})\b'
    matches = re.findall(pattern, text[:500])
    
    if matches:
        prefix, number = matches[0]
        return f"{prefix}{number}"
    
    return None

def remove_cover_sheet(text):
    """
    Remove cover sheet content using layered heuristics.
    
    Strategies (applied in order):
    1) Minerva-specific: Searches for horizontal rules or TOC header in first 3000 chars
    2) Major page break: 6+ consecutive blank lines
    3) Early horizontal rule: 5+ underscores in first 2000 chars
    4) Course code fallback: Skips 4 lines after finding course code
    """
    text = text.replace('\r\n', '\n')

    if 'Minerva University' in text[:500]:
        early_section = text[:3000]
        rule_match = re.search(r'\n\s*_{5,}\s*\n', early_section)
        toc_match = re.search(r'(?i)\n\s*(table of )?contents\s*\n', early_section)
        candidates = []
        if rule_match:
            candidates.append(('rule', rule_match.start(), rule_match.end()))
        if toc_match:
            candidates.append(('toc', toc_match.start(), toc_match.end()))
        if candidates:
            kind, start, end = min(candidates, key=lambda x: x[1])
            return text[start:] if kind == 'toc' else text[end:]

    page_break_pattern = r'(\n\s*){6,}'
    match = re.search(page_break_pattern, text)
    if match:
        return text[match.end():]

    horizontal_rule_pattern = r'\n\s*_{5,}\s*\n'
    match = re.search(horizontal_rule_pattern, text[:2000])
    if match:
        return text[match.end():]

    course_code_pattern = r'\b([A-Z]{2})[\s-]?(\d{2,3})\b'
    match = re.search(course_code_pattern, text[:500])
    if match:
        pos = match.end()
        for _ in range(4):
            next_newline = text.find('\n', pos)
            if next_newline == -1:
                break
            pos = next_newline + 1
        return text[pos:]

    return text

def remove_hc_tags(text):
    """
    Remove HC tags (#tag: explanation) and standalone horizontal rules.
    
    Removes:
    - HC tag blocks: #tag: explanation (continues until blank line)
    - Horizontal rules: Lines with 5+ underscores
    """
    text = re.sub(r'\#[A-Za-z0-9\-]+:[^\n]*(?:\n(?!\n)[^\n]*)*', '', text)
    text = re.sub(r'^\s*_{5,}\s*$', '', text, flags=re.MULTILINE)
    return text

def remove_toc(text):
    """
    Remove table of contents blocks using two strategies.
    
    Strategy 1 - Explicit header: Matches "Table of Contents" or "Contents" header.
                 Ends at double newline or horizontal rule.
    Strategy 2 - Page number heuristic: Detects 3+ lines ending with digits in first 20 lines.
                 Removes up to first double blank line, or first blank line as fallback.
    """
    text = text.replace('\r\n', '\n')

    toc_pattern = r'(?i)(table of )?contents\s*\n'
    match = re.search(toc_pattern, text)
    if match:
        toc_start = match.start()
        remaining = text[match.end():]
        end_match = re.search(r'\n\s*\n|\n\s*_{5,}\s*\n', remaining)
        if end_match:
            return text[:toc_start] + remaining[end_match.end():].lstrip()
        return text[:toc_start]

    lines = text.split('\n')
    sample = lines[:20]
    toc_like = [ln for ln in sample if re.search(r'\s+\d+\s*$', ln)]
    if len(toc_like) >= 3:
        for i in range(len(sample) - 1):
            if sample[i].strip() == '' and sample[i + 1].strip() == '':
                return '\n'.join(lines[i + 1:]).lstrip()
        for i in range(len(sample)):
            if sample[i].strip() == '':
                return '\n'.join(lines[i + 1:]).lstrip()

    return text

def extract_body_only(text):
    """
    Extract main body text by cutting at the earliest end-of-body marker.
    
    Detects: References, Bibliography, Appendix, Appendices, Word Count, AI Statement variants.
    Protection: Only cuts if marker appears after 100+ chars (prevents removing entire document).
    """
    text = text.replace('\r\n', '\n')
    
    end_markers = [
        r'(?i)^(references|bibliography)',
        r'(?i)^(appendix|appendices)',
        r'(?i)^(word count|ai statement|ai use|statement on ai)',
    ]
    
    earliest_match = None
    earliest_pos = len(text)
    
    for pattern in end_markers:
        match = re.search(pattern, text, re.MULTILINE)
        if match and match.start() > 100 and match.start() < earliest_pos:
            earliest_pos = match.start()
            earliest_match = match
    
    if earliest_match:
        return text[:earliest_match.start()].strip()
    
    return text.strip()


def preprocess_dataframe(df):
    """
    Apply all preprocessing steps to create cleaned text columns.
    
    Columns created:
    - text_raw: Original text (baseline)
    - no_cover: Cover sheet + TOC removed
    - text_clean: Body text only (no references, appendix, AI statements, HC tags)
    - course_code: Extracted course code
    - label: 0 = pre-AI courses, 1 = post-AI courses
    """
    df_processed = df.copy()
    df_processed['course_code'] = df_processed['text_raw'].apply(extract_course_code)
    df_processed['no_cover'] = df_processed['text_raw'].apply(remove_cover_sheet).apply(remove_toc)
    df_processed['text_clean'] = df_processed['no_cover'].apply(extract_body_only).apply(remove_hc_tags)
    PRE_AI_COURSES = ['CX51', 'EA51', 'FA51', 'GL92', 'MC51', 
                      'CX50', 'EA50', 'FA50', 'GL91', 'MC50', 'IL199']
    df_processed['label'] = df_processed['course_code'].apply(
        lambda x: 0 if x in PRE_AI_COURSES else 1
    )
    
    return df_processed

# Apply preprocessing and keep relevant columns
df_processed = preprocess_dataframe(df)
df_processed = df_processed[['name', 'course_code', 'label', 'text_raw', 'no_cover', 'text_clean']]

print("Preprocessing complete.")
print(f"Total documents: {len(df_processed)}")

# Display summary table with truncated text for readability
display_df = df_processed.head().copy()
text_cols = ['text_raw', 'no_cover', 'text_clean']
for col in text_cols:
    display_df[col] = display_df[col].apply(lambda x: (x[:75] + '...') if isinstance(x, str) and len(x) > 75 else x)
display(display_df[['course_code', 'name', 'label'] + text_cols])

Preprocessing complete.
Total documents: 51


,course_code,name,label,text_raw,no_cover,text_clean
0,CP192,Reflection on Track Options [final],1,﻿Reflection on Track Options\r\n\r\n\r\n\r\n\r...,Reflection on Track Options\nProcess Documenta...,Reflection on Track Options\nProcess Documenta...
1,SS111,LBA [final],1,﻿LBA: Analyzing the Bangle Market of Charminar...,LBA: Analyzing the Bangle Market of Charminar\...,LBA: Analyzing the Bangle Market of Charminar\...
2,GL96,Elevation Reflection & Engagement [final],1,﻿Elevation Reflection & Engagement \r\nGL96\r\...,One way I plan to engage meaningfully with Hyd...,One way I plan to engage meaningfully with Hyd...
3,GL96,Ethnographic Assignment [final],1,﻿Ethnographic Assignment\r\n\r\n\r\n\r\n\r\nMi...,Ethnographic Assignment\nIntroduction\nFor thi...,Ethnographic Assignment\nIntroduction\nFor thi...
4,SS156,Final assignment [final],1,﻿Tab 1\r\n\r\n\r\n\r\n\r\n\r\n\r\nComparative ...,Comparative Analysis of Political Systems: Fur...,Comparative Analysis of Political Systems: Fur...


In [ ]:
# Manual inspection of all documents
# Commented out to reduce bulk

# pd.set_option('display.max_colwidth', None)

# for idx, row in df_processed.iterrows():
#     print(f"\n{'='*100}")
#     print(f"📄 Document #{idx + 1}: {row['name']}")
#     print(f"   Course: {row['course_code']} | Label: {row['label']} ({'pre-AI' if row['label'] == 0 else 'post-AI'})")
#     print(f"{'='*100}")
        
#     print(f"\n[TEXT_CLEAN - First 100 chars]")
#     print(row['text_clean'][:100] if isinstance(row['text_clean'], str) else row['text_clean'])
#     print(f"\n[TEXT_CLEAN - Last 100 chars]")
#     print(row['text_clean'][-100:] if isinstance(row['text_clean'], str) else row['text_clean'])


📄 Document #1: Reflection on Track Options [final]
   Course: CP192 | Label: 1 (post-AI)

[TEXT_CLEAN - First 100 chars]
Reflection on Track Options
Process Documentation
To ideate what kind of track to propose, I answere

[TEXT_CLEAN - Last 100 chars]
 meetings, as is the case in traditional capstone advising.
HC/LO Applications for the Work Product


📄 Document #2: LBA [final]
   Course: SS111 | Label: 1 (post-AI)

[TEXT_CLEAN - First 100 chars]
LBA: Analyzing the Bangle Market of Charminar
        Located in the “Old City” of Hyderabad, India,

[TEXT_CLEAN - Last 100 chars]
s firms restrict output below the efficient level in order to sustain prices above marginal cost.[3]

📄 Document #3: Elevation Reflection & Engagement [final]
   Course: GL96 | Label: 1 (post-AI)

[TEXT_CLEAN - First 100 chars]
One way I plan to engage meaningfully with Hyderabad is through its local climbing community. I bega

[TEXT_CLEAN - Last 100 chars]
e me to pay attention to differences rather than projec

## Analysis

## Model Selection 

## Training 

In [6]:
#@title also session 5 PCW (CHANGE LATER)

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import metrics, naive_bayes

labels, texts = df['v1'], df['v2']
vectorizer = text.TfidfVectorizer()
vec_texts = vectorizer.fit_transform(texts)


# We train a model
model = naive_bayes.MultinomialNB()
model.fit(vec_texts, labels)
pred_labels = model.predict(vec_texts)

confusion_mat = metrics.confusion_matrix(labels, pred_labels)
sns.heatmap(confusion_mat, square=True, annot=True, fmt='d', cbar=False)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

# some checks to see how the model is doing
print(model.classes_)
print(df['v1'].value_counts())
print(vec_texts.shape) 




KeyError: 'v1'

## Predictions

## Discussion 

## Summary 

## References 

- Session 5 PCW: https://forum.minerva.edu/app/courses/3804/sections/13026/classes/99436 